# Federated traffic forecasting under a MEASURED V2X link

The one experiment where this thesis's **ML** half and its **networking** half produce a single table:
prediction accuracy and communication cost measured together.

* Protocol: FedAvg over sensor-partitioned RSU clients, following **CNFGNN** (Meng et al., KDD 2021).
* Difference from the literature: CNFGNN and FedGTP cost communication *analytically* (params x clients x rounds).
  Here each model update is **dropped and delayed using the distribution measured from the official Eclipse MOSAIC
  ITS-G5 stack** on the Dhaka network (delivery 0.96-0.98, mean 30 ms, p95 57 ms).

Runtime → **GPU**. METR-LA with 8 clients x 30 rounds ≈ 15-25 min on an A100/L4.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import pathlib, subprocess, sys, json, shutil
DRIVE = pathlib.Path('/content/drive/MyDrive/dhakaflow/predict')
REPO  = pathlib.Path('/content/thesis')
!nvidia-smi -L

## 1. Code + measured link statistics

The module and the MOSAIC measurement files come from the repo. Push them from your Mac first:

```bash
rclone copy predict/federated_forecast.py gdrive2tb:dhakaflow/predict/
rclone copy network/results gdrive2tb:dhakaflow/network/results --include "penetration_p*_s*.json"
```

In [ ]:
(REPO/'predict').mkdir(parents=True, exist_ok=True)
(REPO/'network/results').mkdir(parents=True, exist_ok=True)
shutil.copy2(DRIVE/'federated_forecast.py', REPO/'predict/federated_forecast.py')
src = pathlib.Path('/content/drive/MyDrive/dhakaflow/network/results')
n = 0
for f in src.glob('penetration_p*_s*.json'):
    shutil.copy2(f, REPO/'network/results'/f.name); n += 1
print('measured V2X files:', n)
assert n, 'no penetration_*.json on Drive — push them with rclone (see the cell above)'
%cd {REPO}
!python predict/federated_forecast.py --self-test

## 2. Data

METR-LA prepared by the BasicTS notebook is reused if present; otherwise the raw `.h5` is read directly.
The module wants an `.npz` with `values` (T x N) or `data`.

In [ ]:
# Any registered city — the loader is the only place that knows about file formats.
import shutil, pathlib
for city in ('METR-LA', 'PEMS-BAY'):
    dst = REPO/'predict/data/cities'/city; dst.mkdir(parents=True, exist_ok=True)
    src = DRIVE/'raw_data'/city
    if src.exists():
        for f in src.iterdir(): shutil.copy2(f, dst/f.name)
!pip -q install tables
!python predict/city_datasets.py --list

## 3. Run the four arms (centralized / federated / federated+measured-V2X / local-only)

In [ ]:
CITY = 'METR-LA'          # or 'PEMS-BAY', 'PEMS04', 'DHAKA-TWIN' — same command, any city
!python predict/city_datasets.py --city {CITY}
!python predict/federated_forecast.py --city {CITY} --clients 8 --rounds 30 --local-epochs 1 --seed 11

## 4. Communication-budget sweep — the accuracy-vs-cost curve (CNFGNN Fig. 4 shape)

In [ ]:
# Accuracy vs communication cost, across cities (the CNFGNN Fig. 4 shape, on a measured link)
import itertools, json, pandas as pd, shutil
rows = []
for city, clients, rounds in itertools.product(['METR-LA', 'PEMS-BAY'], [4, 8, 16], [10, 30]):
    !python predict/federated_forecast.py --city {city} --clients {clients} --rounds {rounds} --seed 11
    f = REPO/f'predict/results/federated_{city}.json'
    if not f.exists(): continue
    r = json.loads(f.read_text())
    for arm, v in r['arms'].items():
        rows.append(dict(city=city, clients=clients, rounds=rounds, arm=arm, mae=round(v['mae'], 4),
                         MB=round(v['bytes']/1e6, 1), lost=v.get('updates_lost', 0)))
    (DRIVE/'results').mkdir(exist_ok=True)
    shutil.copy2(f, DRIVE/f'results/federated_{city}_c{clients}_r{rounds}.json')
df = pd.DataFrame(rows); df.to_csv(DRIVE/'results/federated_sweep.csv', index=False); display(df)

## 5. Pull back to the repo

```bash
rclone copy gdrive2tb:dhakaflow/predict/results predict/results --include "federated*"
python results/build_results.py
```